This notebook was run on Google Colab, you can test it here:

https://colab.research.google.com/drive/10AAJaNgTRKMWtctb3CUDQm1NWBrYYWAj?usp=sharing

# **Easy/Medium DIfficulaty features**

In this nootebook, I created 9 easy/medium difficulty features. They were tested by training a Random Forest Classifier. It achieved an accuracy of 64.5% when tested on 172 games and trained only on data of the past 2.5 years (older data is still used to calculate features for the train and val games). However, when using other seeds for RFC, accuracy can decrease of a few percents (e. g. 62.5%).

**Imports**

In [84]:
!pip install xgboost

from google.colab import drive
import pandas as pd
import numpy as np
from datetime import date
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [85]:
drive.mount('/content/drive') #Specific to colab

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Importing data and preparing it for models.**

In [86]:
df = pd.read_csv('/content/drive/MyDrive/Book/epl-training.csv')
df = df.drop(5700)

#Convert date to number of days until 31/01/2026 (date when the games are going to be played)
df['Date'] = pd.to_datetime(df['Date'], format="%d/%m/%Y")
target = pd.to_datetime("31/01/2026", format="%d/%m/%Y")
df['Date'] = (target - df['Date']).dt.days
df=df.sort_values("Date", ascending=True)

#Map teams to a number list
le1 = LabelEncoder()
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).astype(str)
le1.fit(all_teams)
df['HomeTeam'] = le1.transform(df['HomeTeam'].astype(str))
df['AwayTeam'] = le1.transform(df['AwayTeam'].astype(str))

#Map referees to a number list
le2 = LabelEncoder()
le2.fit(df['Referee'])
df['Referee'] = le2.transform(df['Referee'].astype(str))

#Map final result to a number list
mapping = {"H":0,"D":1,"A":2 }
df['FTR']=df['FTR'].map(mapping)
df['HTR']=df['HTR'].map(mapping)

#Convert to float
df=df.astype(float)

In [87]:
#Creates a new data frame with only the relevant data which will be directly used
#for training (the features will be calculated from the old df,
#new df is only for training and val)
df_new = df[['Date', 'FTR']].copy()

**First feature: H2H**

This corresponds to a parameter representing how well the home team performed against the away team in the past. There is a coefficient associated to the date of each game between the two teams (high for recent games and low for older). This coefficient is decaying exponentially with the time passed since the game and its parameter k is selected by testing various values and picking the best performing one.

In [88]:
#Finds all the past games between the two teams
def get_past_matches(df, home, away, date):
    mask = (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) | ((df['HomeTeam'] == away) & (df['AwayTeam'] == home))) & (df['Date'] > date)
    return df[mask]

#computes h2h for a specific game
def compute_h2h(df, home, away, date, k=0.001, j=0.8, l=0.1):
  matches = get_past_matches(df, home, away, date)
  if matches.empty: #default value in case no games happened before
      return 0.5

  weighted_sum = 0
  count=0

  #Goes through each games which happend before this one to compute its past h2h
  for _, row in matches.iterrows():

      age_days = row['Date'] - date
      w = np.exp(-k * age_days)

      # 1 point for a win, 0.5 for draw and 0 for loss
      #j is a factor giving higher weight for games played at home for the home team and more weight for away games for the away team
      if row['HomeTeam'] == home:
          if row['FTR'] == 0:   res = 1.0
          elif row['FTR'] == 1: res = 0.5
          else: res = 0.0
          weighted_sum += res * w
          count+=w
      #l gives a better score for the team if the team was the away side in a specific past game
      else:
          if row['FTR'] == 2:   res = 1.0
          elif row['FTR'] == 1: res = 0.5 + l
          else: res = 0.0
          weighted_sum += res * w * j
          count+=w*j

  return weighted_sum/count

#Initialise the column
df_new["H2H"] = 0.0



**Feature 2**

Global recent form: computes a score given by the number of W/D/L in recent games for each of the two team

**Feature 3**

Same thing but with goal difference: weighted average


Here the weight is calculated using a logistic function 1/(1+exp(-kt)), which is more suitable than exponential.




In [89]:
#This function returns the K last games of a team before a specific fixture
def get_last_matches(df, team, date, K):
  mask = (((df['HomeTeam'] == team) | (df['AwayTeam'] == team)) &
          (df['Date'] > date))

  past = df[mask].head(K)
  return past

In [90]:
#This computes the weight given to a game when calculating those features
#k and m are parameters tuned arbitrarily
#zmax is used to avoid overflow for games that happened too long ago
def logistic_weight(age_days, k, m, zmax=500):
    z = k * (age_days - m)
    z = np.clip(z, -zmax, zmax)
    return 1 / (1 + np.exp(z))

#Computes recent form, similar way to feature 1
#Takes the 10 last games and apply weights
def compute_recent_form(df, team, date, K=10, k=0.15, m=30, j=0.8, l=0.1):
    past = get_last_matches(df, team, date, K)
    #default values
    if past.empty:
        return 0.5, 0

    weighted_sum = 0
    weight_total = 0

    gd_weighted_sum = 0
    gd_weight_total = 0

    for _, row in past.iterrows():
        age_days = row['Date'] - date
        w=logistic_weight(age_days, k, m)

        if row['HomeTeam'] == team:
          if row['FTR'] == 0:   res = 1.0
          elif row['FTR'] == 1: res = 0.5
          else: res = 0.0
          weighted_sum += res * w
          weight_total += w

          gd = row['FTHG'] - row['FTAG']
          gd_weighted_sum += gd * w
          gd_weight_total += w

        else:
          if row['FTR'] == 2:   res = 1.0 + l
          elif row['FTR'] == 1: res = 0.5 + l
          else: res = l
          weighted_sum += res * w * j
          weight_total += w*j

          gd = row['FTAG'] - row['FTHG']
          gd_weighted_sum += gd * w*j
          gd_weight_total += w*j

    if weight_total == 0:
        return 0.5, 0.0

    recent_form = weighted_sum / weight_total
    recent_gd = gd_weighted_sum / gd_weight_total if gd_weight_total > 0 else 0.0

    return recent_form, recent_gd




**Feature 4**

Home win rate in the past year

**Feature 5**

Away win rate in the past year

In [91]:
def compute_home_win_rate(df, team, date):
    mask = (
        (df["HomeTeam"] == team)
        & (df["Date"] > date)
        & ((df["Date"] - date) < 365)
    )
    past = df[mask]
    if past.empty:
        return 0.5
    wins = (past["FTR"] == 0).sum()
    return wins / len(past)


def compute_away_win_rate(df, team, date):
    mask = (
        (df["AwayTeam"] == team)
        & (df["Date"] > date)
        & ((df["Date"] - date) < 365)
    )
    past = df[mask]
    if past.empty:
        return 0.5
    wins = (past["FTR"] == 2).sum()
    return wins / len(past)

**Features 67 8 and 9**

Simple statistics:

- gf: Goals scored by the team in the past 10 games

- ga: Goals against by the team in the past 10 games

- sf: Shots scored by the team in the past 10 games

- sa: Shots against by the team in the past 10 games

In [92]:
def compute_recent_simple_stats(df, team, date, K=10):
    past = get_last_matches(df, team, date, K)
    if past.empty:
        return 0.0, 0.0, 0.0, 0.0

    home = past[past["HomeTeam"] == team]
    away = past[past["AwayTeam"] == team]

    gf = home["FTHG"].sum() + away["FTAG"].sum()
    ga = home["FTAG"].sum() + away["FTHG"].sum()
    sf = home["HS"].sum()   + away["AS"].sum()
    sa = home["AS"].sum()   + away["HS"].sum()

    n = len(past)
    return gf / n, ga / n, sf / n, sa / n

**Computing the new features**

In [93]:
#Initialising the new colums of the df used for training the model

df_new["RecentWH"] = 0.0
df_new["RecentWA"] = 0.0
df_new["RecentGDH"] = 0.0
df_new["RecentGDA"] = 0.0
df_new["HomeWinRate"] = 0.0
df_new["AwayWinRate"] = 0.0

df_new["AvgGF_H"] = 0.0
df_new["AvgGA_H"] = 0.0
df_new["AvgSF_H"] = 0.0
df_new["AvgSA_H"] = 0.0

df_new["AvgGF_A"] = 0.0
df_new["AvgGA_A"] = 0.0
df_new["AvgSF_A"] = 0.0
df_new["AvgSA_A"] = 0.0

In [94]:
# Caluclating the features for each row
# This code is really slow, can take up to 2 minutes to run
# Could be massively optimised
for i, game in df.iterrows():
    df_new.loc[i, "H2H"] = compute_h2h(df, game["HomeTeam"], game["AwayTeam"], game["Date"])
    df_new.loc[i, "RecentWH"], df_new.loc[i, "RecentGDH"] = compute_recent_form(df, game["HomeTeam"], game["Date"])
    df_new.loc[i, "RecentWA"], df_new.loc[i, "RecentGDA"] = compute_recent_form(df, game["AwayTeam"], game["Date"])
    df_new.loc[i, "HomeWinRate"] = compute_home_win_rate(df, game["HomeTeam"], game["Date"])
    df_new.loc[i, "AwayWinRate"] = compute_away_win_rate(df, game["AwayTeam"], game["Date"])
    df_new.loc[i, ["AvgGF_A", "AvgGA_A", "AvgSF_A", "AvgSA_A"]] = compute_recent_simple_stats(df, game["AwayTeam"], game["Date"])
    df_new.loc[i, ["AvgGF_H", "AvgGA_H", "AvgSF_H", "AvgSA_H"]] = compute_recent_simple_stats(df, game["HomeTeam"], game["Date"])


In [95]:
# This gives more features to the model, such as form difference
# They can appear redundant since it's only doing simple arithmetics on existing features but is useful for models such as binary trees
df_new["FormDiff"]         = df_new["RecentWH"]  - df_new["RecentWA"]
df_new["GDDiff"]           = df_new["RecentGDH"] - df_new["RecentGDA"]
df_new["ShotDiff"]         = df_new["AvgSF_H"]   - df_new["AvgSF_A"]
df_new["ShotAllowedDiff"]  = df_new["AvgSA_H"]   - df_new["AvgSA_A"]
df_new["AttackBalance"]    = (df_new["AvgGF_H"] + df_new["AvgGF_A"]) - (df_new["AvgGA_H"] + df_new["AvgGA_A"])
df_new["DefenseBalance"]   = (df_new["AvgGA_H"] + df_new["AvgGA_A"])

In [96]:
# Just for visualisation
df_new.sample(10)

,Date,FTR,H2H,RecentWH,RecentWA,RecentGDH,RecentGDA,HomeWinRate,AwayWinRate,AvgGF_H,...,AvgGF_A,AvgGA_A,AvgSF_A,AvgSA_A,FormDiff,GDDiff,ShotDiff,ShotAllowedDiff,AttackBalance,DefenseBalance
2102,7333.0,1.0,0.358173,0.580152,0.898451,0.506006,2.136427,0.833333,0.666667,1.5,...,2.5,0.6,15.4,11.0,-0.318299,-1.630422,-5.1,-1.8,2.5,1.5
9540,290.0,0.0,0.623595,1.016423,0.633940,2.120638,-0.369835,0.588235,0.411765,1.9,...,1.8,1.2,12.9,11.9,0.382483,2.490473,-1.0,0.4,0.7,3.0
5309,4287.0,0.0,0.117864,0.656476,0.515895,0.541696,-0.282881,0.222222,0.157895,1.2,...,1.2,1.7,13.0,13.4,0.140581,0.824577,1.6,4.3,-0.8,3.2
7020,2596.0,2.0,0.255374,0.433212,0.869212,-0.395287,1.043095,0.500000,0.684211,1.3,...,1.7,0.9,11.8,12.9,-0.436000,-1.438383,1.8,-2.1,1.0,2.0
1603,7777.0,0.0,0.612632,0.667744,0.284452,0.258838,-0.510528,0.421053,0.157895,1.1,...,1.0,1.6,11.6,13.0,0.383291,0.769366,-1.6,0.0,-0.9,3.0
105,9226.0,0.0,0.500000,0.719615,0.490671,0.328601,-1.526369,0.600000,0.400000,0.8,...,1.1,2.1,7.2,14.5,0.228944,1.854970,3.3,-1.4,-1.3,3.2
1938,7448.0,2.0,1.000000,0.404091,0.202027,-0.626052,-0.898726,0.611111,0.100000,1.1,...,0.9,1.5,10.3,11.3,0.202063,0.272674,-1.7,1.5,-1.1,3.1
4002,5506.0,0.0,1.000000,0.931995,0.552239,1.378777,0.275548,0.894737,0.333333,2.1,...,1.5,1.0,12.5,11.8,0.379756,1.103229,0.5,-2.6,2.0,1.6
3496,5950.0,0.0,0.682867,0.916259,0.250189,2.506492,-0.602189,0.631579,0.250000,2.9,...,0.9,1.1,9.8,14.4,0.666070,3.108681,7.8,-4.4,1.2,2.6
7327,2282.0,2.0,0.557705,0.332068,0.457801,-0.616263,-0.581751,0.526316,0.263158,1.2,...,0.6,1.5,9.2,14.4,-0.125733,-0.034512,2.2,-1.5,-1.1,2.9


**Preparing the data for training**

In [117]:
#Only train the model on the past 2.5 years of games as old games don't necessarily have enough prior data and are less relevant
max_days=365*2.5
temp=df_new.copy()
df_new = temp[temp["Date"] <= max_days].copy()

#Split dataset between train and val (80/20) with a fixed seed
train_df, val_df = train_test_split(df_new, test_size=0.2, random_state=42)

#Split predict target and rest of data
xtrain = train_df.drop(columns=['FTR'])
ytrain = train_df['FTR']

xval = val_df.drop(columns=['FTR'])
yval = val_df['FTR']

**Random Forest classifier**

Simple model to test if the features are useful
We predict the final result using all the features created.

**Results**

Accuracy: 0.6453488372093024

              precision    recall  f1-score   support

         0.0       0.60      0.81      0.69        72
         1.0       0.82      0.30      0.44        47
         2.0       0.66      0.74      0.70        53

    accuracy                           0.65       172
    macro avg      0.70      0.61      0.61       172
    weighted avg   0.68      0.65      0.62       172

In [122]:
model = RandomForestClassifier(class_weight='balanced',
    n_estimators=200,
    random_state=42
)

model.fit(xtrain, ytrain)

ypred=model.predict(xval)
acc=accuracy_score(yval, ypred)
print(acc)
print(classification_report(yval, ypred))

0.6453488372093024
              precision    recall  f1-score   support

         0.0       0.60      0.81      0.69        72
         1.0       0.82      0.30      0.44        47
         2.0       0.66      0.74      0.70        53

    accuracy                           0.65       172
   macro avg       0.70      0.61      0.61       172
weighted avg       0.68      0.65      0.62       172



**XGB**

Worse than random forest in this case

**Results**

Accuracy : 0.6395348837209303
              precision    recall  f1-score   support

         0.0       0.65      0.76      0.70        72
         1.0       0.63      0.36      0.46        47
         2.0       0.63      0.72      0.67        53

    accuracy                           0.64       172
    macro avg      0.64      0.61      0.61       172
    weighted avg   0.64      0.64      0.63       172


In [119]:
model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(xtrain, ytrain)

ypred = model.predict(xval)

acc = accuracy_score(yval, ypred)
print("Accuracy :", acc)
print(classification_report(yval, ypred))


Accuracy : 0.6395348837209303
              precision    recall  f1-score   support

         0.0       0.65      0.76      0.70        72
         1.0       0.63      0.36      0.46        47
         2.0       0.63      0.72      0.67        53

    accuracy                           0.64       172
   macro avg       0.64      0.61      0.61       172
weighted avg       0.64      0.64      0.63       172



This should be ignored. I tried to change the RFC to a RFR to predict a goal difference then classify it to a final result H/D/A instead of directly predicting the final result. A similar method was used by participants in NCAA. This showed poor results for our case, probably because football games have way less goals than basketball.

In [100]:
"""max_days=365*10
df_new = df_new[df_new["Date"] <= max_days].copy()

#Split dataset between train and val (80/20) with a fixed seed
train_df, val_df = train_test_split(df_new, test_size=0.2, random_state=42)

#Split predict target and rest of data
xtrain = train_df.drop(columns=['FTR', 'GD'])
ytrain = train_df['GD']
FTR_train=train_df['FTR']
xval = val_df.drop(columns=['FTR', 'GD'])
yval = val_df['GD']
FTR_val=val_df['FTR']

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(xtrain, ytrain)
gd_pred = model.predict(xval)

print("MAE:", mean_absolute_error(yval, gd_pred))

pred_FTR = []
for gd in gd_pred:
    if gd >= 0:
        pred_FTR.append(0.0)  # home win
    elif gd <= -0:
        pred_FTR.append(2.0)  # away win
    else:
        pred_FTR.append(1.0)  # draw

pred_FTR = np.array(pred_FTR)
print("Accuracy:", accuracy_score(FTR_val, pred_FTR))
print(classification_report(FTR_val, pred_FTR))"""

'max_days=365*10\ndf_new = df_new[df_new["Date"] <= max_days].copy()\n\n#Split dataset between train and val (80/20) with a fixed seed\ntrain_df, val_df = train_test_split(df_new, test_size=0.2, random_state=42)\n\n#Split predict target and rest of data\nxtrain = train_df.drop(columns=[\'FTR\', \'GD\'])\nytrain = train_df[\'GD\']\nFTR_train=train_df[\'FTR\']\nxval = val_df.drop(columns=[\'FTR\', \'GD\'])\nyval = val_df[\'GD\']\nFTR_val=val_df[\'FTR\']\n\nmodel = RandomForestRegressor(\n    n_estimators=200,\n    random_state=42,\n    n_jobs=-1\n)\n\nmodel.fit(xtrain, ytrain)\ngd_pred = model.predict(xval)\n\nprint("MAE:", mean_absolute_error(yval, gd_pred))\n\npred_FTR = []\nfor gd in gd_pred:\n    if gd >= 0:\n        pred_FTR.append(0.0)  # home win\n    elif gd <= -0:\n        pred_FTR.append(2.0)  # away win\n    else:\n        pred_FTR.append(1.0)  # draw\n\npred_FTR = np.array(pred_FTR)\nprint("Accuracy:", accuracy_score(FTR_val, pred_FTR))\nprint(classification_report(FTR_val, p